In [1]:
import pandas as pd

clv = pd.read_csv('../data/processed/customer_clv.csv')
repeat_model = pd.read_csv('../data/processed/repeat_purchase_model.csv')

# join on customer_unique_id
combined = clv.merge(
    repeat_model[['customer_unique_id', 'delivery_delay_days', 'review_score', 'repeat_purchase_probability', 'will_repeat']],
    on='customer_unique_id', how='inner'
)

print(combined.shape)
combined.head()

(93357, 13)


,customer_unique_id,frequency,recency,T,monetary_value,predicted_clv_12m,days_since_last_purchase,clv_tier,avg_order_value,delivery_delay_days,review_score,repeat_purchase_probability,will_repeat
0,0000366f3b9a7992bf8c76cfdf3221e2,0.0,0.0,112.0,0.0,13.104614,112.0,Platinum,141.90,-5.0,5.0,0.471419,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,0.0,0.0,115.0,0.0,12.962604,115.0,Gold,27.19,-5.0,4.0,0.459387,0
2,0000f46a3911fa3c0805444483337064,0.0,0.0,538.0,0.0,5.313159,538.0,Bronze,86.22,-2.0,3.0,0.483357,0
3,0000f6ccb0745a6a4b88665a16c9f078,0.0,0.0,322.0,0.0,7.548574,322.0,Silver,43.62,-12.0,4.0,0.511677,0
4,0004aac84e0df4da2b147fca70cf8255,0.0,0.0,289.0,0.0,8.074530,289.0,Silver,196.89,-8.0,5.0,0.493487,0


In [2]:
# bucket delivery delay into simple bands
combined['delay_band'] = pd.cut(
    combined['delivery_delay_days'],
    bins=[-200, -14, 0, 14, 200],
    labels=['Arrived 14+ days early', 'Arrived early', 'Arrived late', 'Arrived 14+ days late']
)

combined.groupby('delay_band')['will_repeat'].agg(['mean', 'count'])

C:\Users\bumba\AppData\Local\Temp\ipykernel_5964\3316042614.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  combined.groupby('delay_band')['will_repeat'].agg(['mean', 'count'])


,mean,count
delay_band,,
Arrived 14+ days early,0.031916,40638
Arrived early,0.029029,46367
Arrived late,0.024161,5008
Arrived 14+ days late,0.027530,1344


In [3]:
combined.groupby(['clv_tier', 'delay_band'], observed=True)['will_repeat'].agg(['mean', 'count'])

mean  count
clv_tier delay_band                             
Bronze   Arrived 14+ days early  0.047619  12201
         Arrived early           0.053686  10431
         Arrived late            0.050459    654
         Arrived 14+ days late   0.042553    188
Gold     Arrived 14+ days early  0.018486   8439
         Arrived early           0.017407  12294
         Arrived late            0.018491   2055
         Arrived 14+ days late   0.027692    650
Platinum Arrived 14+ days early  0.036462   9983
         Arrived early           0.030012  12162
         Arrived late            0.041841    956
         Arrived 14+ days late   0.096774     93
Silver   Arrived 14+ days early  0.019571  10015
         Arrived early           0.018031  11480
         Arrived late            0.007446   1343
         Arrived 14+ days late   0.004843    413

In [4]:
segment_summary = combined.groupby('clv_tier', observed=True).agg(
    customers=('customer_unique_id', 'count'),
    avg_predicted_clv=('predicted_clv_12m', 'mean'),
    total_predicted_clv=('predicted_clv_12m', 'sum'),
    repeat_rate=('will_repeat', 'mean'),
    avg_delivery_delay=('delivery_delay_days', 'mean'),
    avg_review_score=('review_score', 'mean')
).reset_index()

segment_summary

,clv_tier,customers,avg_predicted_clv,total_predicted_clv,repeat_rate,avg_delivery_delay,avg_review_score
0,Bronze,23474,5.973143,140213.551224,0.050354,-13.719690,4.236943
1,Gold,23438,11.035134,258641.469597,0.018176,-9.673735,4.003285
2,Platinum,23194,19.091101,442799.006371,0.033543,-12.674054,4.289795
3,Silver,23251,8.289375,192736.257494,0.017849,-11.347727,4.107243


In [5]:
platinum_value = segment_summary.loc[segment_summary['clv_tier'] == 'Platinum', 'total_predicted_clv'].values[0]
print(f"Total 12-month predicted CLV concentrated in Platinum tier: R${platinum_value:,.0f}")
print(f"Share of total customer base: {segment_summary.loc[segment_summary['clv_tier']=='Platinum','customers'].values[0] / segment_summary['customers'].sum():.1%}")

Total 12-month predicted CLV concentrated in Platinum tier: R$442,799
Share of total customer base: 24.8%


In [6]:
segment_summary

,clv_tier,customers,avg_predicted_clv,total_predicted_clv,repeat_rate,avg_delivery_delay,avg_review_score
0,Bronze,23474,5.973143,140213.551224,0.050354,-13.719690,4.236943
1,Gold,23438,11.035134,258641.469597,0.018176,-9.673735,4.003285
2,Platinum,23194,19.091101,442799.006371,0.033543,-12.674054,4.289795
3,Silver,23251,8.289375,192736.257494,0.017849,-11.347727,4.107243
